# Graph Programming — 03: Breadth-First Search (BFS)

**BFS** explores nodes level by level — all neighbors first, then their neighbors.  
Think of it as: *go wide before going deep.*

```
Graph:       BFS from 0 visits: 0 → 1,2 → 3,4 → 5
      0
     / \
    1   2
   / \   \
  3   4   5

Level 0: [0]
Level 1: [1, 2]
Level 2: [3, 4, 5]
```

## When to use BFS
- Find the **shortest path** in an unweighted graph
- Find **minimum steps** to reach a target
- Level-order traversal
- "Spread" problems (rotting oranges, walls and gates)
- Bipartite graph check

## BFS vs DFS
| | BFS | DFS |
|---|---|---|
| Data structure | Queue (deque) | Stack (or recursion) |
| Traversal | Level by level | Branch by branch |
| Shortest path? | YES (unweighted) | No |
| Memory | More (stores whole level) | Less |

---

In [ ]:
from collections import deque, defaultdict

# ============================================================
# BFS TEMPLATE (memorize this)
# ============================================================
#
#   from collections import deque
#   queue = deque([start])
#   visited = {start}
#   while queue:
#       node = queue.popleft()     # FIFO = breadth-first
#       for neighbor in graph[node]:
#           if neighbor not in visited:
#               visited.add(neighbor)
#               queue.append(neighbor)

graph = defaultdict(list, {
    0: [1, 2],
    1: [0, 3, 4],
    2: [0, 5],
    3: [1],
    4: [1],
    5: [2]
})

def bfs(graph, start):
    visited = {start}
    queue = deque([start])
    order = []

    while queue:
        node = queue.popleft()    # popleft = FIFO (breadth-first)
        order.append(node)
        for neighbor in graph[node]:
            if neighbor not in visited:
                visited.add(neighbor)
                queue.append(neighbor)

    return order

print("BFS from 0:", bfs(graph, 0))
# [0, 1, 2, 3, 4, 5]  — level by level

---
## Problem 1: Shortest Path in Unweighted Graph

BFS gives you the shortest path **for free** because it visits nodes level by level.  
Level = number of edges = distance.

**Key:** Track distance alongside each node in the queue.

In [ ]:
def shortest_path(graph, start, end):
    if start == end:
        return 0

    visited = {start}
    queue = deque([(start, 0)])    # (node, distance)

    while queue:
        node, dist = queue.popleft()
        for neighbor in graph[node]:
            if neighbor == end:
                return dist + 1    # found it!
            if neighbor not in visited:
                visited.add(neighbor)
                queue.append((neighbor, dist + 1))

    return -1  # no path exists

print(shortest_path(graph, 0, 5))   # 2 (0→2→5)
print(shortest_path(graph, 0, 3))   # 2 (0→1→3)
print(shortest_path(graph, 3, 5))   # 4 (3→1→0→2→5)

---
## Problem 2: BFS on a Grid — Shortest Path

LC 1091 — Shortest Path in Binary Matrix.

Given an `n x n` grid of 0s and 1s, find the shortest clear path (0s only) from top-left to bottom-right.  
You can move in **8 directions** (including diagonals).

In [ ]:
def shortest_path_binary_matrix(grid):
    n = len(grid)
    if grid[0][0] == 1 or grid[n-1][n-1] == 1:
        return -1

    # 8 directions (including diagonals)
    DIRS = [(-1,-1),(-1,0),(-1,1),(0,-1),(0,1),(1,-1),(1,0),(1,1)]

    queue = deque([(0, 0, 1)])   # (row, col, path_length)
    visited = {(0, 0)}

    while queue:
        r, c, dist = queue.popleft()
        if r == n-1 and c == n-1:
            return dist
        for dr, dc in DIRS:
            nr, nc = r+dr, c+dc
            if 0 <= nr < n and 0 <= nc < n and grid[nr][nc] == 0 and (nr,nc) not in visited:
                visited.add((nr, nc))
                queue.append((nr, nc, dist+1))

    return -1

grid1 = [[0,1],[1,0]]
grid2 = [[0,0,0],[1,1,0],[1,1,0]]
print(shortest_path_binary_matrix(grid1))  # 2
print(shortest_path_binary_matrix(grid2))  # 4

---
## Problem 3: Rotting Oranges (LC 994) — Multi-Source BFS

Grid: `0` = empty, `1` = fresh orange, `2` = rotten orange.  
Every minute, rotten oranges rot all adjacent fresh oranges.  
Find minimum minutes to rot all oranges, or -1 if impossible.

**Key insight:** Start BFS from **all** rotten oranges simultaneously = multi-source BFS.

In [ ]:
def oranges_rotting(grid):
    rows, cols = len(grid), len(grid[0])
    fresh = 0
    queue = deque()

    # Seed queue with ALL rotten oranges at once (multi-source BFS)
    for r in range(rows):
        for c in range(cols):
            if grid[r][c] == 2:
                queue.append((r, c, 0))   # (row, col, time)
            elif grid[r][c] == 1:
                fresh += 1

    if fresh == 0:
        return 0

    DIRS = [(0,1),(0,-1),(1,0),(-1,0)]
    time = 0

    while queue:
        r, c, t = queue.popleft()
        for dr, dc in DIRS:
            nr, nc = r+dr, c+dc
            if 0 <= nr < rows and 0 <= nc < cols and grid[nr][nc] == 1:
                grid[nr][nc] = 2        # rot it
                fresh -= 1
                time = t + 1
                queue.append((nr, nc, t+1))

    return time if fresh == 0 else -1

print(oranges_rotting([[2,1,1],[1,1,0],[0,1,1]]))  # 4
print(oranges_rotting([[2,1,1],[0,1,1],[1,0,1]]))  # -1 (isolated fresh)
print(oranges_rotting([[0,2]]))                    # 0

---
## Problem 4: Word Ladder (LC 127) — BFS on Implicit Graph

Transform `beginWord` → `endWord` one letter at a time. Each intermediate word must be in `wordList`.  
Find the minimum number of steps.

This is a **shortest path** problem where each word is a node, and two words are connected if they differ by exactly one letter.

In [ ]:
def word_ladder(beginWord, endWord, wordList):
    word_set = set(wordList)
    if endWord not in word_set:
        return 0

    queue = deque([(beginWord, 1)])   # (word, steps)
    visited = {beginWord}

    while queue:
        word, steps = queue.popleft()

        # Try changing each character to a-z
        for i in range(len(word)):
            for ch in 'abcdefghijklmnopqrstuvwxyz':
                new_word = word[:i] + ch + word[i+1:]
                if new_word == endWord:
                    return steps + 1
                if new_word in word_set and new_word not in visited:
                    visited.add(new_word)
                    queue.append((new_word, steps + 1))

    return 0

print(word_ladder("hit", "cog", ["hot","dot","dog","lot","log","cog"]))  # 5
# hit → hot → dot → dog → cog

---
## BFS Level-by-Level Pattern

When you need to process each level separately (e.g., binary tree level order), use this variant:

In [ ]:
def bfs_by_level(graph, start):
    """Returns list of levels, each level is a list of nodes."""
    visited = {start}
    queue = deque([start])
    levels = []

    while queue:
        level_size = len(queue)      # how many nodes are at current level
        level = []

        for _ in range(level_size):  # process only this level's nodes
            node = queue.popleft()
            level.append(node)
            for neighbor in graph[node]:
                if neighbor not in visited:
                    visited.add(neighbor)
                    queue.append(neighbor)

        levels.append(level)

    return levels

print("Levels from 0:", bfs_by_level(graph, 0))
# [[0], [1, 2], [3, 4, 5]]

---
## Summary

```
BFS Cheatsheet:

  from collections import deque
  queue = deque([start])
  visited = {start}
  steps = 0

  while queue:
      for _ in range(len(queue)):     # process level-by-level
          node = queue.popleft()
          if node == target: return steps
          for neighbor in graph[node]:
              if neighbor not in visited:
                  visited.add(neighbor)
                  queue.append(neighbor)
      steps += 1
```

**Next:** `04_islands.ipynb` — Island Problems (DFS + BFS applied to grids)